In [1]:
import torch
import sqlite3
import pandas as pd
from pathlib import Path
from PIL import Image
from transformers import Blip2Processor, Blip2ForConditionalGeneration

/home/agrupa-lab/agrupa/IE_capstones/Omar/venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DTYPE = torch.float16

print(f"Device: {DEVICE}")
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

BASE_DIR = Path("/home/agrupa-lab/agrupa")
DB_PATH = BASE_DIR / "agrupa.sqlite"
OUTPUT_DIR = Path("/home/agrupa-lab/agrupa/IE_capstones/Omar/outputs")
OUTPUT_DIR.mkdir(exist_ok=True)

Device: cuda
GPU: NVIDIA GeForce RTX 5090
VRAM: 33.7 GB


In [3]:
conn = sqlite3.connect(DB_PATH)

# Get P artworks that have images
query = """
    SELECT a.cat_no, a.titulo, a.autor, a.is_fauna, a.is_religious, a.century,
           i.file_path
    FROM artwork a
    INNER JOIN artwork_image i ON a.cat_no = i.cat_no
    WHERE substr(a.cat_no, 1, 1) = 'P'
"""
df = pd.read_sql(query, conn)
conn.close()

print(f"Total P artworks with images: {len(df)}")
print(f"Fauna: {df['is_fauna'].sum()}")
print(f"Non-fauna: {(df['is_fauna'] == 0).sum()}")
df.head()

Total P artworks with images: 6266
Fauna: 2275
Non-fauna: 3991


,cat_no,titulo,autor,is_fauna,is_religious,century,file_path
0,P000002,El juicio de Paris,"Albani, Francesco",1,0,17th c.,/home/agrupa-lab/agrupa/scripts/urls/obras/P00...
1,P000006,Sagrada Familia y el cardenal Fernando de Medici,"Allori, Alessandro",1,1,16th c.,/home/agrupa-lab/agrupa/scripts/urls/obras/P00...
2,P000013,Rendición de Sevilla al rey san Fernando,"Flipart, Charles-Joseph",1,0,18th c.,/home/agrupa-lab/agrupa/scripts/urls/obras/P00...
3,P000014,"María Isabel de Borbón y Sajonia, infanta de N...","Ruta, Clemente",1,0,18th c.,/home/agrupa-lab/agrupa/scripts/urls/obras/P00...
4,P000015,La Anunciación,"Angelico, Fra",1,1,15th c.,/home/agrupa-lab/agrupa/scripts/urls/obras/P00...


In [4]:
df['image_exists'] = df['file_path'].apply(lambda x: Path(x).exists())

print(f"Images found on disk: {df['image_exists'].sum()}")
print(f"Images missing on disk: {(~df['image_exists']).sum()}")

# Keep only artworks where image actually exists
df_ready = df[df['image_exists']].reset_index(drop=True)
print(f"\nReady for MLLM processing: {len(df_ready)}")

Images found on disk: 6266
Images missing on disk: 0

Ready for MLLM processing: 6266


In [5]:
print("Loading BLIP-2 model... (this may take a few minutes)")

MODEL_NAME = "Salesforce/blip2-opt-2.7b"

processor = Blip2Processor.from_pretrained(MODEL_NAME)
model = Blip2ForConditionalGeneration.from_pretrained(
    MODEL_NAME,
    torch_dtype=DTYPE,
    device_map="auto"
)
model.eval()

print("Model loaded successfully!")
print(f"VRAM used: {torch.cuda.memory_allocated() / 1e9:.2f} GB")

Loading BLIP-2 model... (this may take a few minutes)


The image processor of type `BlipImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 
Loading weights: 100%|██████████| 1247/1247 [00:01<00:00, 1178.98it/s]


Model loaded successfully!
VRAM used: 7.70 GB


In [6]:
def generate_description(image_path, prompt=None):
    try:
        image = Image.open(image_path).convert("RGB")
        
        if prompt:
            inputs = processor(image, text=prompt, return_tensors="pt").to(DEVICE, DTYPE)
        else:
            inputs = processor(image, return_tensors="pt").to(DEVICE, DTYPE)
        
        with torch.no_grad():
            output = model.generate(
                **inputs,
                max_new_tokens=150,
                num_beams=5,
                min_length=20
            )
        
        description = processor.decode(output[0], skip_special_tokens=True)
        return description.strip()
    
    except Exception as e:
        return f"ERROR: {str(e)}"

In [7]:
# Test on 5 random artworks before running the full pipeline
sample = df_ready.sample(5, random_state=42)

print("Running smoke test on 5 artworks...\n")
for _, row in sample.iterrows():
    desc = generate_description(row['file_path'])
    print(f"Artwork: {row['titulo']}")
    print(f"Author:  {row['autor']}")
    print(f"Description: {desc}")
    print("-" * 60)

Running smoke test on 5 artworks...

Artwork: La reina María Luisa de Parma
Author:  Goya y Lucientes, Francisco de (Copia)
Description: a painting of a woman wearing a hat and dress
------------------------------------------------------------
Artwork: Sagrada Familia
Author:  Cambiaso, Luca
Description: a painting of the madonna and child with two men
------------------------------------------------------------
Artwork: San Ambrosio imponiendo el hábito a san Agustín
Author:  García Hidalgo, José
Description: a painting of a group of people in a church
------------------------------------------------------------
Artwork: Vieja mesándose los cabellos
Author:  Massys, Jan (Círculo de)
Description: a painting of an old woman with her hands on her head
------------------------------------------------------------
Artwork: La Felicidad Eterna (boceto)
Author:  Madrazo y Agudo, José de
Description: a painting of a woman holding an olympic torch
-----------------------------------------------

In [ ]:
PROMPT = "Describe this artwork in detail, including the people or animals depicted, their appearance, actions, social roles, and the overall mood of the scene."

print("Running prompted smoke test on 5 artworks...\n")
for _, row in sample.iterrows():
    desc = generate_description(row['file_path'], prompt=PROMPT)
    print(f"Artwork: {row['titulo']}")
    print(f"Author:  {row['autor']}")
    print(f"Description: {desc}")
    print("-" * 60)

Running prompted smoke test on 5 artworks...

Artwork: La reina María Luisa de Parma
Author:  Goya y Lucientes, Francisco de (Copia)
Description: Describe this artwork in detail, including the people or animals depicted, their appearance, actions, social roles, and the overall mood of the scene.
------------------------------------------------------------
Artwork: Sagrada Familia
Author:  Cambiaso, Luca
Description: Describe this artwork in detail, including the people or animals depicted, their appearance, actions, social roles, and the overall mood of the scene.
------------------------------------------------------------
Artwork: San Ambrosio imponiendo el hábito a san Agustín
Author:  García Hidalgo, José
Description: Describe this artwork in detail, including the people or animals depicted, their appearance, actions, social roles, and the overall mood of the scene.
------------------------------------------------------------
Artwork: Vieja mesándose los cabellos
Author:  Massys, J

: 

Adding a prompt to the BLIP-2 model does not work, it is a pure captioning model from what I can see, it can produce good short descriptions, but they are not very detailed.